# Challenge 2: Final Model and Predictions (Team Seven)

Predicting whether an online shopping visit ends in a purchase (`Revenue=True`). Scored on **F1 for the positive class**.

This notebook shows the whole path from the raw files to `predictions.csv`:

1. Load `challenge2_train.csv` and make the team's shared split (80% development, 20% hold-out).
2. Recap the eight models the team evaluated (each has its own notebook with the full tuning work).
3. Compare the **shortlisted** models once on the hold-out. The settings below were tuned by grid search in those notebooks, using only the development rows and 5-fold cross validation.
4. Pick the final model with a rule that is written down *before* the hold-out is scored.
5. Refit the final model on all training rows and predict `challenge2_test.csv`.

The test file is only opened in step 5. Run it from the folder that contains both CSV files; every random step uses seed 42.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix)

RANDOM_STATE = 42
NAVY, ORANGE, TEAL, RED, GRAY = "#1f3b73", "#f28e2b", "#2a9d8f", "#d1495b", "#9aa0a6"
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__)

## 1. Load and split

Same code, split and seed as `challenge2_validation_split.ipynb` and every model notebook.

In [ ]:
train = pd.read_csv("challenge2_train.csv")

def to_01(col):
    """Turn True/False (bool or text) into 1/0."""
    if col.dtype == bool:
        return col.astype(int)
    return col.astype(str).str.strip().str.lower().map({"true": 1, "false": 0})

y = to_01(train["Revenue"])
X = train.drop(columns=["ID", "Revenue"]).copy()
X["Weekend"] = to_01(X["Weekend"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
assert train.shape == (9247, 19) and len(X_train) == 7397 and len(X_val) == 1850
print(f"Development: {len(X_train):,} rows ({y_train.mean():.1%} buyers) | "
      f"Hold-out: {len(X_val):,} rows ({y_val.mean():.1%} buyers)")

## 2. The eight models the team evaluated

Each was tuned with grid search on the development rows (settings are in the linked notebook). Scores are 5-fold cross validation on the development rows, copied from each notebook's own summary. Models with a tuned threshold use the best-F1 cut-off from out-of-fold predictions; the decision tree and random forest use the default 0.5 cut-off. These are development scores, so they are slightly optimistic and are not directly comparable to the hold-out numbers in step 3.

In [ ]:
team_results = pd.DataFrame([
    ["Logistic regression", "angel_logistic_regression.ipynb", 0.927, 0.700, 0.690, "0.58"],
    ["Naive Bayes",         "angel_naive_bayes.ipynb",          0.900, 0.668, 0.686, "0.30"],
    ["K-nearest neighbors", "challenge2_knn.ipynb",             0.922, 0.735, 0.679, "0.34"],
    ["Neural network",      "challenge2_neural_network.ipynb",  0.932, 0.751, 0.694, "0.38"],
    ["SVM (RBF)",           "challenge2_svm_gradient_boosting_DIEGO.ipynb", 0.912, 0.736, 0.700, "-0.563 (decision score)"],
    ["Gradient boosting",   "challenge2_svm_gradient_boosting_DIEGO.ipynb", 0.937, 0.765, 0.704, "0.30"],
    ["Decision tree",       "challenge2_decision_tree_random_forest_DAVID.ipynb", 0.924, 0.702, 0.678, "0.50 (default)"],
    ["Random forest",       "challenge2_decision_tree_random_forest_DAVID.ipynb", 0.935, 0.764, 0.705, "0.50 (default)"],
], columns=["model", "notebook", "CV ROC-AUC", "CV PR-AUC", "CV F1", "threshold"]).set_index("model")
team_results.sort_values("CV F1", ascending=False)

## 3. Shortlist and hold-out comparison

**Shortlist (chosen before scoring the hold-out):** the top models on development ROC-AUC, PR-AUC and F1, plus one average of the two best:
gradient boosting, random forest, neural network, SVM, and the average of gradient boosting and random forest.

The extra experiment in `challenge2_gradient_boosting_enhanced.ipynb` (a wider grid and four engineered features) changed development F1 by only +0.002, which is inside the fold-to-fold spread (0.02 to 0.03), so the team's simpler gradient boosting settings are used.

**Selection rule (fixed in advance):** take the model with the highest hold-out F1. If other models are within 0.01 of it, take the one among them with the best hold-out PR-AUC.

**Threshold rule (same for every model):** a model's cut-off is the middle of the range of cut-offs whose out-of-fold F1 is within 0.005 of the best, which is steadier than the single best spike. It is chosen using only the rows the model is being fitted on.

In [ ]:
log_cols = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
            "ProductRelated", "ProductRelated_Duration", "PageValues"]
rate_cols = ["BounceRates", "ExitRates", "SpecialDay"]
cat_cols = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
bin_cols = ["Weekend"]
assert set(log_cols + rate_cols + cat_cols + bin_cols) == set(X.columns)

# Trees: no scaling needed.
tree_prep = ColumnTransformer([
    ("num", "passthrough", log_cols + rate_cols + bin_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
# Neural network and SVM: log, then scale, the skewed counts and durations.
scaled_prep = ColumnTransformer([
    ("log", Pipeline([("log1p", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]), log_cols),
    ("rate", StandardScaler(), rate_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

# Settings come from each model's own tuning notebook.
SPECS = {
    "Gradient boosting": dict(prep=tree_prep, score="proba", model=lambda: GradientBoostingClassifier(
        random_state=RANDOM_STATE, learning_rate=0.03, max_depth=4, n_estimators=100, subsample=1.0)),
    "Random forest": dict(prep=tree_prep, score="proba", model=lambda: RandomForestClassifier(
        random_state=RANDOM_STATE, n_estimators=300, max_depth=None, min_samples_leaf=4,
        max_features=0.5, class_weight="balanced", n_jobs=-1)),
    "Neural network": dict(prep=scaled_prep, score="proba", model=lambda: MLPClassifier(
        random_state=RANDOM_STATE, hidden_layer_sizes=(32, 16), alpha=1.0, learning_rate_init=0.01,
        max_iter=1000, early_stopping=True)),
    "SVM (RBF)": dict(prep=scaled_prep, score="decision", model=lambda: SVC(
        kernel="rbf", C=30, gamma=0.01, random_state=RANDOM_STATE)),
}
CANDIDATES = {
    "Gradient boosting": ["Gradient boosting"],
    "Random forest": ["Random forest"],
    "Neural network": ["Neural network"],
    "SVM (RBF)": ["SVM (RBF)"],
    "GB + RF average": ["Gradient boosting", "Random forest"],
}

def make_pipe(spec):
    return Pipeline([("prep", spec["prep"]), ("model", spec["model"]())])

def get_score(pipe, spec, data):
    return pipe.predict_proba(data)[:, 1] if spec["score"] == "proba" else pipe.decision_function(data)

def fit_base(name, data, target):
    """5-fold out-of-fold scores (same folds as the team notebooks), plus a model fitted on all rows."""
    spec = SPECS[name]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(target))
    for tr, va in cv.split(data, target):
        fold_model = make_pipe(spec).fit(data.iloc[tr], target.iloc[tr])
        oof[va] = get_score(fold_model, spec, data.iloc[va])
    return {"spec": spec, "oof": oof, "model": make_pipe(spec).fit(data, target)}

def percentile(reference, scores):
    """Where each score falls in the sorted reference scores (0 to 1). Puts different models on one scale."""
    ref = np.sort(reference)
    return np.searchsorted(ref, scores, side="right") / len(ref)

def candidate_oof(parts, bases):
    if len(parts) == 1:
        return bases[parts[0]]["oof"]
    return np.mean([percentile(bases[p]["oof"], bases[p]["oof"]) for p in parts], axis=0)

def candidate_score(parts, bases, data):
    scores = [get_score(bases[p]["model"], bases[p]["spec"], data) for p in parts]
    if len(parts) == 1:
        return scores[0]
    return np.mean([percentile(bases[p]["oof"], s) for p, s in zip(parts, scores)], axis=0)

def pick_threshold(oof, target, tol=0.005):
    grid = np.unique(np.quantile(oof, np.linspace(0.60, 0.95, 176)))
    f1s = np.array([f1_score(target, oof >= t) for t in grid])
    return float(np.median(grid[f1s >= f1s.max() - tol]))

In [ ]:
# Fit every base model on the development rows only.
dev_bases = {name: fit_base(name, X_train, y_train) for name in SPECS}

# Score the hold-out ONCE per candidate.
rows, holdout_pred, holdout_score = [], {}, {}
for cand, parts in CANDIDATES.items():
    dev_oof = candidate_oof(parts, dev_bases)
    thr = pick_threshold(dev_oof, y_train)
    score = candidate_score(parts, dev_bases, X_val)
    pred = (score >= thr).astype(int)
    holdout_pred[cand], holdout_score[cand] = pred, score
    rows.append({
        "model": cand,
        "threshold": thr,
        "dev F1 (out-of-fold)": f1_score(y_train, dev_oof >= thr),
        "hold-out F1": f1_score(y_val, pred),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred),
        "ROC-AUC": roc_auc_score(y_val, score),
        "PR-AUC": average_precision_score(y_val, score),
        "predicted buyers": int(pred.sum()),
    })
holdout = pd.DataFrame(rows).set_index("model").sort_values("hold-out F1", ascending=False)
holdout.round(3)

The hold-out has 1,850 rows and about 287 buyers, so its F1 is noisy. A bootstrap over the hold-out rows shows how much:

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
y_val_arr = y_val.to_numpy()
boot_idx = rng.integers(0, len(y_val_arr), size=(1000, len(y_val_arr)))
ci = {}
for cand, pred in holdout_pred.items():
    f1s = [f1_score(y_val_arr[i], pred[i]) for i in boot_idx]
    ci[cand] = np.percentile(f1s, [2.5, 97.5])

order = list(holdout.index)
fig, ax = plt.subplots(figsize=(9, 4))
vals = holdout["hold-out F1"].to_numpy()
lo = vals - np.array([ci[c][0] for c in order])
hi = np.array([ci[c][1] for c in order]) - vals
ax.bar(order, vals, yerr=[lo, hi], capsize=5, color=NAVY)
for i, v in enumerate(vals):
    ax.text(i, 0.30, f"{v:.3f}", ha="center", color="white", fontweight="bold")
ax.set_ylim(0.25, 0.85)
ax.set_ylabel("hold-out F1 (buyer class)")
ax.set_title("Hold-out F1 with 95% bootstrap interval")
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## 4. Select the final model

In [ ]:
best_f1 = holdout["hold-out F1"].max()
contenders = holdout[holdout["hold-out F1"] >= best_f1 - 0.01]
FINAL_NAME = contenders["PR-AUC"].idxmax()
FINAL_PARTS = CANDIDATES[FINAL_NAME]

print(f"Best hold-out F1: {best_f1:.3f}")
print("Within 0.01 of the best:", ", ".join(contenders.index))
print(f"Selected by the rule: {FINAL_NAME}")
holdout.to_csv("final_holdout_comparison.csv")

In [ ]:
pred = holdout_pred[FINAL_NAME]
cm = confusion_matrix(y_val, pred)
share = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.8, 4))
ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}\n({share[i, j]:.0%})", ha="center", va="center", fontsize=12,
                color="white" if share[i, j] > 0.5 else "black", fontweight="bold")
ax.set_xticks([0, 1], ["no purchase", "purchase"]); ax.set_yticks([0, 1], ["no purchase", "purchase"])
ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.grid(False)
ax.set_title(f"{FINAL_NAME}: hold-out\nF1 {f1_score(y_val, pred):.3f}")
plt.tight_layout(); plt.show()

## 5. Refit on all training rows and predict the test set

From here the hold-out is no longer needed, so the selected model is refitted on **all 9,247 training rows**. Its out-of-fold predictions over those rows set the cut-off with the same rule as before. `challenge2_test.csv` is opened here for the first time.

In [ ]:
full_bases = {name: fit_base(name, X, y) for name in FINAL_PARTS}
full_oof = candidate_oof(FINAL_PARTS, full_bases)
final_threshold = pick_threshold(full_oof, y)
print(f"Final model: {FINAL_NAME}")
print(f"Cut-off from all-training out-of-fold predictions: {final_threshold:.4f}")
print(f"Out-of-fold F1 on all 9,247 training rows at that cut-off: {f1_score(y, full_oof >= final_threshold):.3f}")

In [ ]:
test = pd.read_csv("challenge2_test.csv")
X_test = test.drop(columns=["ID"]).copy()
X_test["Weekend"] = to_01(X_test["Weekend"])
X_test = X_test[X.columns]   # same columns, same order as training

test_score = candidate_score(FINAL_PARTS, full_bases, X_test)
test_pred = test_score >= final_threshold

predictions = pd.DataFrame({"ID": test["ID"], "Revenue": test_pred})
predictions.to_csv("predictions.csv", index=False)

# Checks on the file that will be submitted.
check = pd.read_csv("predictions.csv")
assert list(check.columns) == ["ID", "Revenue"]
assert len(check) == len(test) == 3083
assert check["ID"].is_unique and (check["ID"].to_numpy() == test["ID"].to_numpy()).all()
assert check["Revenue"].isin([True, False]).all() and check.notna().all().all()
print(f"predictions.csv: {len(check):,} rows | predicted buyers {int(check['Revenue'].sum()):,} "
      f"({check['Revenue'].mean():.1%}; the training data is {y.mean():.1%})")
check.head()

## Summary

- Eight models were tuned and evaluated in the team notebooks; five were shortlisted for the hold-out.
- The final model was chosen by the rule written above, using the hold-out table in step 3.
- The hold-out was scored once per shortlisted model and never used for tuning.
- `predictions.csv` holds `ID` and `Revenue` (True/False) for every row of `challenge2_test.csv`, in the original order.